# Satellite tasking: the operations problem that fits

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/benchmarks/satellite-tasking.ipynb)

Reproduces the benchmark published at [zksf.org/applications/space](https://zksf.org/applications/space/).

An imaging satellite has more observation requests than it can serve. Overlapping windows conflict. Choose the schedule of maximum value.

This is a maximum weighted independent set, which needs **one qubit per request**. That linear encoding is why this problem is testable on a quantum computer and vehicle routing is not.


In [ ]:
!pip install -q qiskit qiskit-aer ortools scipy numpy

## The instance

In [ ]:
import numpy as np, itertools, time
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from ortools.sat.python import cp_model

SEED, N = 20260902, 14
rng = np.random.default_rng(SEED + N)
start, dur, value = rng.uniform(0, 100, N), rng.uniform(4, 14, N), rng.uniform(1, 10, N)
conflict = np.zeros((N, N), int)
for i in range(N):
    for j in range(i + 1, N):
        if start[i] < start[j] + dur[j] and start[j] < start[i] + dur[i]:
            conflict[i, j] = conflict[j, i] = 1
print(f"{N} requests, {conflict.sum()//2} conflicting pairs, {N} qubits needed")

## Two classical baselines

Exhaustive search proves the optimum. CP-SAT is what a mission planner would actually use, and it is the fair comparison.

In [ ]:
def feasible(x):
    return not any(x[i] and x[j] and conflict[i, j] for i in range(N) for j in range(i+1, N))

t0 = time.perf_counter()
exact = max(float(value @ np.array(x)) for x in itertools.product([0,1], repeat=N) if feasible(x))
t_exact = time.perf_counter() - t0

m = cp_model.CpModel(); xs = [m.NewBoolVar(f"x{i}") for i in range(N)]
for i in range(N):
    for j in range(i+1, N):
        if conflict[i, j]: m.Add(xs[i] + xs[j] <= 1)
m.Maximize(sum(int(round(value[i]*1000)) * xs[i] for i in range(N)))
s = cp_model.CpSolver(); t0 = time.perf_counter(); s.Solve(m); t_cp = time.perf_counter() - t0
print(f"exhaustive {exact:.3f} in {t_exact:.3f}s")
print(f"CP-SAT     {s.ObjectiveValue()/1000:.3f} in {t_cp:.4f}s   <- integer-scaled, hence the 3rd decimal")

## QAOA

In [ ]:
pen = float(value.max()) * 2
Q = np.zeros((N, N)); np.fill_diagonal(Q, -value)
for i in range(N):
    for j in range(i+1, N):
        if conflict[i, j]: Q[i, j] = Q[j, i] = pen / 2

sim, evals = AerSimulator(), {"n": 0}
def best_sample(params, p):
    evals["n"] += 1
    Qs = (Q + Q.T) / 2
    qc = QuantumCircuit(N); qc.h(range(N))
    h = [-Qs[i,i]/2 - sum(Qs[i,j] for j in range(N) if j!=i)/4 for i in range(N)]
    for g, b in zip(params[:p], params[p:]):
        for i in range(N):
            for j in range(i+1, N):
                if abs(Qs[i,j]) > 1e-12: qc.rzz(2*g*Qs[i,j]/4, i, j)
            if abs(h[i]) > 1e-12: qc.rz(2*g*h[i], i)
        for i in range(N): qc.rx(2*b, i)
    qc.measure_all()
    out = -np.inf
    for bits in sim.run(qc, shots=512).result().get_counts():
        x = np.array([int(c) for c in reversed(bits)], int)
        if feasible(x): out = max(out, float(value @ x))
    return out

rng2 = np.random.default_rng(SEED + N)
t0 = time.perf_counter(); found = -np.inf
for r in range(3):
    x0 = np.r_[np.full(2, 0.6), np.full(2, 0.4)] if r == 0 else rng2.uniform(0, np.pi, 4)
    res = minimize(lambda t: -min(1e6, best_sample(t, 2)), x0, method="COBYLA",
                   options={"maxiter": 150, "rhobeg": 0.4})
    found = max(found, best_sample(res.x, 2))
print(f"QAOA p=2   {found:.3f} in {time.perf_counter()-t0:.1f}s, {evals['n']} evals")
print(f"optimal: {abs(found - exact) < 1e-9}")

## What you should see

CP-SAT solves this in around 10 milliseconds, roughly a thousand times faster than QAOA, and its solve time barely moves as the problem grows while exhaustive search climbs exponentially.

In the published runs QAOA matched the proven optimum in 1 of 3 instances. Raise `N` to 16 and it degrades. That degradation as the instance grows is the thing to watch, because it is a software problem that could be solved long before the hardware arrives.


## Why your numbers will not match exactly

Two reasons, both worth understanding before you compare against the published table.

**Sampling is not seeded.** QAOA reads its answer from measurement counts, so every run explores slightly differently and the optimiser lands somewhere slightly different. Expect the same *shape*, not the same digits: a result in the top couple of percent of feasible solutions, hitting the exact optimum sometimes and not others. If you re-run this cell a few times you will see that spread directly, and that spread is itself the honest finding about QAOA's reliability.

**The published timings came through the ZKSF engine**, which adds job handling, routing and certification around the same simulation. This notebook calls Aer directly, so it is faster here. The algorithm and the answer quality are the same; only the wall clock differs.

The classical numbers do reproduce exactly wherever they are proven optimal, because a proof is not a sample.


## Next

- [The full benchmark page](https://zksf.org/applications/space/), with the analysis and the caveats
- [How we benchmark](https://zksf.org/applications/methodology/): the rules every one of these follows
- [All applications](https://zksf.org/applications/) across six sectors
- [Certification](https://zksf.org/quantum-computing-certification/): what the accuracy statements assert
